## Query Log Compaction — DBpedia (`tab:compaction`)

Downstream application replacing the caching simulation (see
`notebooks/caching/README.md` for why): given a query log, select a small
representative subset ("coreset") that still faithfully stands in for the
whole log's **results**, not just its wording.

**Protocol**

- **Corpus:** the RQ1 DBpedia global corpus (1,677 queries,
  `datasets/dbpedia_testset_RQ1.json`), oracle answer sets from
  `datasets/result_sets_dbpedia.json` (100% coverage of the corpus, verified
  before writing this notebook).
- **Selection:** deterministic agglomerative clustering (average linkage)
  over each model's own cosine-distance matrix, cut to *k* clusters; each
  cluster's medoid becomes its representative. No randomness (unlike
  k-means/k-medoids), and no shared cross-model similarity threshold —
  models are compared at **equal budget** (same *k*), not at a shared
  absolute cosine value. This sidesteps the caching experiment's two open
  problems: comparing differently-calibrated embedding spaces at one fixed
  θ, and a sequential/path-dependent simulation that amplified GPU
  non-determinism run to run.
- **Fidelity:** for every query in the corpus, look up its cluster's medoid
  and compute the true answer-set Jaccard between the query and the medoid.
  A faithful compaction keeps this high even as *k* shrinks.
- **Budget grid:** k/N ∈ {50%, 30%, 20%, 10%, 5%} of the corpus.
- **Models:** identical roster to `tab:caching` — SparqlCE, TF-IDF, BM25,
  E5-base-v2, Qwen2.5-Coder-0.5B (code LLM), CodeBERT, GraphCodeBERT, SPBERT,
SBERT/MiniLM,
  SBERT/MPNet.

Requires the SparqlCE checkpoint `sparqlce/models/final_model_mrr_0.4487.pt`
and the fine-tuned E5 checkpoint `checkpoints/e5_square_dbpedia/best_model`
(neither in git — see the main README). Outputs
`results/compaction/compaction_dbpedia.json`, a fidelity-vs-budget figure,
and ready-to-paste LaTeX rows for `tab:compaction`.

In [ ]:
import gc
import os

# Must be set before the first CUDA call (i.e. before `import torch` touches
# the GPU), so cuBLAS allocates a deterministic workspace.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import normalize

# ---- Project root (works from notebooks/compaction/ or anywhere in the repo) ----
_PROJECT_ROOT = Path.cwd()
while not (_PROJECT_ROOT / "datasets").exists():
    assert _PROJECT_ROOT != _PROJECT_ROOT.parent, "could not locate repo root"
    _PROJECT_ROOT = _PROJECT_ROOT.parent

for _p in (
    _PROJECT_ROOT,
    _PROJECT_ROOT / "notebooks" / "evaluation-RQ1",
    _PROJECT_ROOT / "scripts",
):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))


# ---- Config ----
TESTSET_PATH = _PROJECT_ROOT / "datasets" / "dbpedia_testset_RQ1.json"
RESULTS_PATH = _PROJECT_ROOT / "datasets" / "result_sets_dbpedia.json"

SPARQLCE_CKPT = (
    _PROJECT_ROOT / "sparqlce" / "models" / "final_model_mrr_0.4487.pt"
)

FT_E5_PATH = (
    _PROJECT_ROOT
    / "checkpoints"
    / "e5_square_dbpedia"
    / "best_model"
)

OUT_DIR = _PROJECT_ROOT / "results" / "compaction"

SEED = 42

# Budget grid: fraction of the corpus kept as representatives.
BUDGET_FRACTIONS = [0.50, 0.30, 0.20, 0.10, 0.05]

# Same positive-construction threshold used everywhere else in the paper.
HIGH_FIDELITY_THRESHOLD = 0.8

# Headline budget reported in the main table.
HEADLINE_FRACTION = 0.10

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ---------------------------------------------------------------------------
# Determinism.
#
# SEED only fixes RNG-based initialization -- it does not make CUDA kernels
# bit-reproducible run to run. Repeated executions of the caching notebook
# showed SparqlCE (GATv2 scatter/gather message passing) and both Qwen3
# paths (dense attention kernels) drifting between runs, while every
# sparse/SentenceTransformer baseline stayed bit-identical. Force
# determinism everywhere torch supports it; an op with no deterministic GPU
# kernel raises here instead of silently drifting the clustering below into
# a different coreset. See the SparqlCE encode cell for a CPU fallback.
# ---------------------------------------------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

print(
    f"project root: {_PROJECT_ROOT}\n"
    f"device: {DEVICE}"
)

### 1. Corpus and oracle

Same RQ1 DBpedia global corpus and oracle used throughout the paper (built
with the same helpers as `notebooks/evaluation-RQ1/evaluate_rq1.py`, kept
self-contained here so this notebook runs without a torch-dependent import
of that module). `result_sets_dbpedia.json` was verified to cover 100% of
the 1,677-query corpus before this notebook was written.

In [ ]:
def normalize_query(q):
    """Normalize query formatting for exact oracle lookup only."""
    return " ".join(q.split())


def freeze_json(value):
    """Recursively convert a JSON value into a hashable form (result rows
    are dicts/lists of variable bindings, which frozenset() cannot hold
    directly)."""
    if isinstance(value, dict):
        return tuple(
            sorted((key, freeze_json(item)) for key, item in value.items())
        )
    if isinstance(value, list):
        return tuple(freeze_json(item) for item in value)
    return value


def get_query_text(x):
    if x is None:
        return ""
    if isinstance(x, dict):
        return str(x.get("query") or x.get("sparql") or "").strip()
    return str(x).strip()


def get_anchor_query(item):
    return get_query_text(
        item.get("anchor_query") or item.get("query") or item.get("base_query")
    )


def iter_labeled_queries(item, positives_only=False):
    out = []
    for p in item.get("positive_queries") or []:
        q = get_query_text(p)
        if q:
            out.append(q)
    if not positives_only:
        for n in item.get("negative_queries") or []:
            q = get_query_text(n)
            if q:
                out.append(q)
    return out


def build_global_corpus(raw_data):
    # Identical construction to evaluate_rq1.py's build_global_corpus, so
    # this is the exact same 1,677-query corpus used for RQ1.
    unique_queries, seen_q = [], set()

    def add_unique(q):
        if q and q not in seen_q:
            seen_q.add(q)
            unique_queries.append(q)

    for item in raw_data:
        anchor_q = get_anchor_query(item)
        if not anchor_q:
            continue
        add_unique(anchor_q)
        for q in iter_labeled_queries(item, positives_only=False):
            add_unique(q)

    return unique_queries, seen_q


def jaccard(a, b):
    union = len(a | b)
    return len(a & b) / union if union else 0.0


with open(TESTSET_PATH, encoding="utf-8") as f:
    raw = json.load(f)

corpus, seen_q = build_global_corpus(raw)

with open(RESULTS_PATH, encoding="utf-8") as f:
    raw_result_sets = json.load(f)

result_sets = {}
for raw_query, rows in raw_result_sets.items():
    if not rows:
        continue
    key = normalize_query(raw_query)
    answer_set = frozenset(freeze_json(row) for row in rows)
    if key in result_sets and result_sets[key] != answer_set:
        raise ValueError(
            "Conflicting oracle result sets after query normalization:\n"
            f"{key}"
        )
    result_sets[key] = answer_set

missing = [q for q in corpus if normalize_query(q) not in result_sets]
if missing:
    raise ValueError(
        f"{len(missing)} corpus queries have no oracle result set, "
        f"e.g.: {missing[:3]}"
    )

true_answers = [result_sets[normalize_query(q)] for q in corpus]

N = len(corpus)
print(f"corpus: {N} queries | oracle coverage: {N - len(missing)}/{N}")

### 2. Embed the corpus once per model

Reuses the same encoders as `notebooks/caching/caching_simulation_dbpedia.ipynb`
and the fine-tuned E5 model (full fine-tune, mean pooling; base model
adapter + last-token pooling + fp32, not `SentenceTransformer(adapter_path)`
— see that module's docstring for the mean-pooling bug this avoids). TF-IDF
and BM25 are fit directly on this corpus: unlike RQ1/caching, there is no
generalization claim here (we are clustering this exact log, not predicting
on unseen queries), so no separate train/test split is needed.

In [ ]:
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sentence_transformers import SentenceTransformer

corpus_embeddings = {}


def clear_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# SparqlCE — trained checkpoint
#
# GATv2's scatter/gather message passing is the one op in this notebook that
# may lack a deterministic CUDA kernel under torch.use_deterministic_algorithms.
# Try the default device first; if that op raises, fall back to CPU (cheap --
# only ~1,700 short queries) rather than silently accept a nondeterministic run.
# ---------------------------------------------------------------------------
from sparqlce.embedder import SPARQLEmbedder
from sparqlce.config import Config as _SparqlCEConfig


def _encode_with_sparqlce(queries, batch_size=32):
    original_device = _SparqlCEConfig.DEVICE
    try:
        embedder = SPARQLEmbedder(str(SPARQLCE_CKPT))
        embs = [
            embedder.encode(queries[i : i + batch_size])
            for i in tqdm(
                range(0, len(queries), batch_size), desc="SparqlCE"
            )
        ]
    except RuntimeError as exc:
        if "deterministic" not in str(exc).lower():
            raise
        print(
            f"SparqlCE GPU encode is non-deterministic ({exc});"
            " falling back to CPU."
        )
        _SparqlCEConfig.DEVICE = torch.device("cpu")
        embedder = SPARQLEmbedder(str(SPARQLCE_CKPT))
        embs = [
            embedder.encode(queries[i : i + batch_size])
            for i in tqdm(
                range(0, len(queries), batch_size), desc="SparqlCE (CPU)"
            )
        ]
    finally:
        _SparqlCEConfig.DEVICE = original_device
    return embedder, embs


sparqlce, embs = _encode_with_sparqlce(corpus)

corpus_embeddings["SparqlCE"] = normalize(
    np.vstack(embs), norm="l2", axis=1
).astype(np.float32, copy=False)

del sparqlce, embs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# TF-IDF
# ---------------------------------------------------------------------------
tfidf = TfidfVectorizer(norm="l2").fit(corpus)
corpus_embeddings["TF-IDF"] = (
    tfidf.transform(corpus).toarray().astype(np.float32, copy=False)
)


# ---------------------------------------------------------------------------
# BM25
#
# Okapi-BM25 term-weight vectors, L2-normalised so cosine distance between
# ANY pair of corpus queries is well-defined -- identical implementation to
# notebooks/caching/caching_simulation_dbpedia.ipynb's BM25Vectorizer (that
# notebook's rationale applies verbatim: rank_bm25's BM25Okapi only scores
# query-vs-corpus, not symmetric pairwise similarity, so it cannot feed a
# clustering step that needs all-pairs distances).
# ---------------------------------------------------------------------------
class BM25Vectorizer:
    """Okapi-BM25 term weighting with optional row normalisation."""

    def __init__(self, *, k1=1.5, b=0.75, norm="l2", **count_vectorizer_kwargs):
        self.k1 = float(k1)
        self.b = float(b)
        self.norm = norm
        self.vectorizer = CountVectorizer(**count_vectorizer_kwargs)

    def fit(self, documents):
        counts = (
            self.vectorizer.fit_transform(documents).tocsr().astype(np.float32)
        )
        n_documents = counts.shape[0]
        if n_documents == 0:
            raise ValueError("BM25Vectorizer cannot be fit on an empty corpus.")

        document_frequency = np.asarray((counts > 0).sum(axis=0)).ravel()
        self.idf_ = np.log(
            1.0 + (n_documents - document_frequency + 0.5) / (document_frequency + 0.5)
        ).astype(np.float32)

        document_lengths = np.asarray(counts.sum(axis=1)).ravel()
        self.avg_document_length_ = float(document_lengths.mean())
        if self.avg_document_length_ <= 0:
            raise ValueError("BM25Vectorizer received only empty documents.")
        return self

    def transform(self, documents):
        counts = self.vectorizer.transform(documents).tocoo().astype(np.float32)
        document_lengths = np.asarray(counts.tocsr().sum(axis=1)).ravel()
        length_factor = (
            1.0 - self.b + self.b * (document_lengths / self.avg_document_length_)
        )
        denominator = counts.data + self.k1 * length_factor[counts.row]
        weighted_data = (
            self.idf_[counts.col] * counts.data * (self.k1 + 1.0) / denominator
        )
        weighted = sparse.csr_matrix(
            (weighted_data, (counts.row, counts.col)),
            shape=counts.shape,
            dtype=np.float32,
        )
        if self.norm is not None:
            weighted = normalize(weighted, norm=self.norm, axis=1, copy=False)
        return weighted

    def fit_transform(self, documents):
        return self.fit(documents).transform(documents)


bm25 = BM25Vectorizer(k1=1.5, b=0.75, norm="l2").fit(corpus)
corpus_embeddings["BM25"] = (
    bm25.transform(corpus).toarray().astype(np.float32, copy=False)
)


# ---------------------------------------------------------------------------
# Dense text-embedding baselines
# ---------------------------------------------------------------------------
model_specs = [
    {
        "name": "E5-base-v2",
        "model_id": "intfloat/e5-base-v2",
        "texts": [f"query: {q}" for q in corpus],
        "batch_size": 32,
    },
    {
        "name": "SBERT/MiniLM",
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "texts": corpus,
        "batch_size": 64,
    },
    {
        "name": "SBERT/MPNet",
        "model_id": "sentence-transformers/all-mpnet-base-v2",
        "texts": corpus,
        "batch_size": 32,
    },
    {
        "name": "CodeBERT",
        "model_id": "microsoft/codebert-base",
        "texts": corpus,
        "batch_size": 64,
    },
    {
        "name": "GraphCodeBERT",
        "model_id": "microsoft/graphcodebert-base",
        "texts": corpus,
        "batch_size": 64,
    },
    {
        "name": "SPBERT",
        "model_id": "razent/spbert-mlm-wso-base",
        "texts": corpus,
        "batch_size": 64,
    },
]

for spec in model_specs:
    model = SentenceTransformer(spec["model_id"], device=DEVICE)
    encoded = model.encode(
        spec["texts"],
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=True,
        batch_size=spec["batch_size"],
    ).astype(np.float32, copy=False)

    corpus_embeddings[spec["name"]] = normalize(
        encoded, norm="l2", axis=1, copy=False
    ).astype(np.float32, copy=False)

    del encoded
    clear_model(model)


# ---------------------------------------------------------------------------
# Qwen2.5-Coder-0.5B — generative code LLM used as an embedder (last-token
# pooling), replacing the dense Qwen3-Embedding-0.6B (off-label; 0.5B ~ scale
# of the 0.6B it replaces).
# ---------------------------------------------------------------------------
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F

CODER_MODEL = "Qwen/Qwen2.5-Coder-0.5B"
CODER_NAME = "Qwen2.5-Coder-0.5B"

def _last_token_pool(h, am):
    if (am[:, -1].sum() == am.shape[0]):
        return h[:, -1]
    lens = am.sum(dim=1) - 1
    idx = torch.arange(h.shape[0], device=h.device)
    return h[idx, lens]

coder_tok = AutoTokenizer.from_pretrained(CODER_MODEL, padding_side="left")
if coder_tok.pad_token_id is None:
    coder_tok.pad_token = coder_tok.eos_token
coder_model = AutoModel.from_pretrained(CODER_MODEL, torch_dtype=torch.float32).to(DEVICE).eval()

@torch.inference_mode()
def encode_coder(queries, batch_size=8, max_length=512):
    out = []
    for start in tqdm(range(0, len(queries), batch_size), desc=CODER_NAME):
        batch = queries[start:start + batch_size]
        tok = coder_tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        h = coder_model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"]).last_hidden_state
        emb = F.normalize(_last_token_pool(h, tok["attention_mask"]).float(), p=2, dim=1)
        out.append(emb.cpu().numpy())
    return np.vstack(out).astype(np.float32, copy=False)

corpus_embeddings[CODER_NAME] = encode_coder(corpus)
del coder_model, coder_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Fine-tuned E5-base-v2 — SQuARE DBpedia full fine-tune (capacity-matched
# fine-tuned baseline; replaces the fine-tuned Qwen3 baseline). Mean pooling,
# "query: " prefix.
# ---------------------------------------------------------------------------
FT_E5_NAME = "E5-base-v2 (fine-tuned)"

if not FT_E5_PATH.exists():
    raise FileNotFoundError(
        f"Fine-tuned E5 model not found: {FT_E5_PATH} — run "
        "notebooks/finetuning/04_e5_contrastive_finetune_square.ipynb first."
    )

ft_e5_tok = AutoTokenizer.from_pretrained(str(FT_E5_PATH))
ft_e5_model = AutoModel.from_pretrained(str(FT_E5_PATH), torch_dtype=torch.float32).to(DEVICE).eval()

def _mean_pool(h, am):
    m = am.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.inference_mode()
def encode_ft_e5(queries, batch_size=32, max_length=512):
    out = []
    for start in tqdm(range(0, len(queries), batch_size), desc=FT_E5_NAME):
        batch = [f"query: {q}" for q in queries[start:start + batch_size]]
        tok = ft_e5_tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        h = ft_e5_model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"]).last_hidden_state
        emb = F.normalize(_mean_pool(h, tok["attention_mask"]).float(), p=2, dim=1)
        out.append(emb.cpu().numpy())
    return np.vstack(out).astype(np.float32, copy=False)

corpus_embeddings[FT_E5_NAME] = encode_ft_e5(corpus)
del ft_e5_model, ft_e5_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Sanity checks
# ---------------------------------------------------------------------------
for name, E in corpus_embeddings.items():
    if E.shape[0] != N:
        raise ValueError(f"{name}: expected {N} rows, got {E.shape[0]}")

    row_norms = np.linalg.norm(E, axis=1)
    nonzero_rows = row_norms > 0
    if np.any(nonzero_rows) and not np.allclose(
        row_norms[nonzero_rows], 1.0, atol=1e-5
    ):
        raise ValueError(f"{name}: non-zero rows are not L2-normalised.")

    print(f"{name:>34}: {E.shape}")

### 3. Coreset selection and fidelity

Deterministic agglomerative clustering (average linkage) over each model's
own cosine-distance matrix, cut to *k* clusters; each cluster's medoid
(the member minimizing total in-cluster distance) becomes the representative
for every query assigned to that cluster. No random initialization, unlike
k-means/k-medoids — the same distance matrix always yields the same
clustering.

In [ ]:
def cosine_distance_matrix(E):
    # Pairwise cosine distance (1 - cosine similarity) for L2-normalised
    # rows, symmetrised and clipped against tiny floating-point noise.
    sims = np.clip(E @ E.T, -1.0, 1.0)
    dist = 1.0 - sims
    dist = (dist + dist.T) / 2.0
    np.fill_diagonal(dist, 0.0)
    return np.clip(dist, 0.0, None)


def medoids_for_clusters(dist, labels):
    # Index of the minimum-total-distance member of each cluster.
    reps = {}
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        if len(idx) == 1:
            reps[c] = int(idx[0])
            continue
        sub = dist[np.ix_(idx, idx)]
        reps[c] = int(idx[np.argmin(sub.sum(axis=1))])
    return reps


def evaluate_budget(E, true_answers, k):
    # Cluster to k groups, assign medoids, and measure how faithfully each
    # query's medoid stands in for its own true answer set.
    dist = cosine_distance_matrix(E)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method="average")
    labels = fcluster(Z, t=k, criterion="maxclust")
    reps = medoids_for_clusters(dist, labels)

    jaccards = np.empty(len(labels))
    for i, c in enumerate(labels):
        jaccards[i] = jaccard(true_answers[i], true_answers[reps[c]])

    return {
        "k_requested": int(k),
        "k_actual": int(labels.max()),
        "mean_fidelity": float(jaccards.mean()),
        "frac_high_fidelity": float((jaccards >= HIGH_FIDELITY_THRESHOLD).mean()),
    }


budget_ks = sorted(
    {max(1, round(N * frac)) for frac in BUDGET_FRACTIONS}, reverse=True
)
print(f"corpus size N={N}, budgets (k): {budget_ks}")

all_results = {}
for name, E in corpus_embeddings.items():
    all_results[name] = {}
    for k in tqdm(budget_ks, desc=name):
        all_results[name][k] = evaluate_budget(E, true_answers, k)
    print(f"done: {name}")

In [ ]:
# ============================================================================
# 4. Save the full grid + display
# ============================================================================
OUT_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "kg": "dbpedia",
    "seed": SEED,
    "corpus_size": N,
    "budget_ks": budget_ks,
    "budget_fractions": BUDGET_FRACTIONS,
    "high_fidelity_threshold": HIGH_FIDELITY_THRESHOLD,
    "results": all_results,
}
out_json = OUT_DIR / "compaction_dbpedia.json"
with open(out_json, "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved {out_json}\n")

rows = []
for name, by_k in all_results.items():
    for k, r in by_k.items():
        rows.append({
            "model": name,
            "k": k,
            "compression": f"{k}/{N}",
            "mean_fidelity": r["mean_fidelity"],
            "frac_high_fidelity": r["frac_high_fidelity"],
        })
df = pd.DataFrame(rows)

print("Mean fidelity (Jaccard to cluster medoid):")
print(df.pivot_table(index="model", columns="k", values="mean_fidelity").round(3))

print(f"\nFraction of queries at high fidelity (Jaccard >= {HIGH_FIDELITY_THRESHOLD}):")
print(df.pivot_table(index="model", columns="k", values="frac_high_fidelity").round(3))

In [ ]:
# ============================================================================
# 5. Fidelity-vs-budget figure
# ============================================================================
import matplotlib.pyplot as plt

order = [
    "SparqlCE",
    "TF-IDF",
    "BM25",
    "E5-base-v2",
    "Qwen2.5-Coder-0.5B",
    "CodeBERT",
    "GraphCodeBERT",
    "SPBERT",
    "E5-base-v2 (fine-tuned)",
    "SBERT/MiniLM",
    "SBERT/MPNet",
]

fig, ax = plt.subplots(figsize=(5, 4))
for name in order:
    xs = [k / N for k in budget_ks]
    ys = [all_results[name][k]["mean_fidelity"] for k in budget_ks]
    style = (
        {"color": "black", "linewidth": 2.5, "zorder": 5}
        if name == "SparqlCE"
        else {"linewidth": 1.2, "alpha": 0.85}
    )
    ax.plot(xs, ys, marker="o", markersize=4, label=name, **style)

ax.set_xlabel("Compression ratio (k / N kept)")
ax.set_ylabel("Mean fidelity (Jaccard to medoid)")
ax.invert_xaxis()
ax.set_ylim(0, 1)
ax.legend(fontsize=7, loc="best")
ax.set_title("Query log compaction: fidelity vs. budget (DBpedia)")
fig.tight_layout()

fig_path = _PROJECT_ROOT / "results" / "compaction_fidelity_dbpedia.pdf"
fig.savefig(fig_path)
print(f"saved {fig_path}")
plt.show()

In [ ]:
# ============================================================================
# 6. LaTeX rows for tab:compaction
# ============================================================================
headline_k = min(budget_ks, key=lambda k: abs(k / N - HEADLINE_FRACTION))

latex_names = {
    "SparqlCE": r"\textsc{SparqlCE}",
    "TF-IDF": "TF-IDF",
    "BM25": r"BM25~\cite{robertson2009bm25}",
    "E5-base-v2": "E5-base-v2",
    "Qwen2.5-Coder-0.5B": "Qwen2.5-Coder-0.5B",
    "CodeBERT": "CodeBERT",
    "GraphCodeBERT": "GraphCodeBERT",
    "SPBERT": "SPBERT",
    "E5-base-v2 (fine-tuned)": "E5-base-v2 (fine-tuned)",
    "SBERT/MiniLM": "SBERT/MiniLM",
    "SBERT/MPNet": "SBERT/MPNet",
}

print(
    f"% tab:compaction — headline budget k={headline_k} "
    f"({headline_k}/{N} = {headline_k / N:.1%} kept), "
    f"full grid in results/compaction/compaction_dbpedia.json"
)

for name in order:
    r = all_results[name][headline_k]
    print(
        f"{latex_names[name]:<28} & {headline_k}/{N} "
        f"& {r['mean_fidelity']:.3f} & {r['frac_high_fidelity']:.3f} \\\\"
    )

best = all_results["SparqlCE"][headline_k]
alts = {name: all_results[name][headline_k] for name in order[1:]}
best_alt = max(alts, key=lambda name: alts[name]["mean_fidelity"])

print("\n% headline sentence draft:")
print(
    f"% At k={headline_k} ({headline_k / N:.0%} of the log kept), "
    f"SparqlCE's coreset retains {best['mean_fidelity']:.2f} mean fidelity "
    f"({best['frac_high_fidelity']:.0%} of queries at Jaccard>={HIGH_FIDELITY_THRESHOLD}), "
    f"vs. {alts[best_alt]['mean_fidelity']:.2f} for the best baseline ({best_alt})."
)

### Notes

- **Why agglomerative clustering, not k-means/k-medoids:** average-linkage
  clustering over a fixed distance matrix is fully deterministic — the same
  embeddings always yield the same clusters and medoids. k-means/k-medoids
  depend on random initialization, which would reintroduce exactly the kind
  of run-to-run variability the caching experiment struggled with.
- **Why budget-matched, not threshold-matched:** every model is clustered to
  the *same number* of representatives at each budget, so no model gets an
  advantage or penalty from having a different "natural" similarity scale —
  this is the fix for the caching experiment's shared-θ-across-differently-
  calibrated-spaces problem.
- **Oracle-free selection:** clustering only ever sees embeddings; the true
  answer sets are used exclusively to *score* fidelity after the fact, never
  to guide which queries get merged.
- Rerunning with a different corpus split or budget grid: change
  `BUDGET_FRACTIONS` / `HEADLINE_FRACTION` — report the values above in the
  paper unless there's a reason to change them.
- Wikidata twin: `datasets/result_sets_wikidata.json` now exists in the repo;
  a `query_log_compaction_wikidata.ipynb` following this exact structure
  would just need `datasets/wikidata_testset_RQ1.json` and the Wikidata
  SparqlCE/E5-ft checkpoints swapped in.